# Image Anti-Spoofing — FakeVsReal_Image_1

**Goal:** Detect whether an image is REAL or FAKE (AI-generated OR deepfake/manipulated), using a comparative
model study - same pattern as the Voice Anti-Spoofing project (Baseline CNN, Deeper CNN, ViT).

Third project in the multi-modal anti-spoofing portfolio, alongside:
- [Voice Anti-Spoofing](https://huggingface.co/LaabhGupta/voice-antispoofing)
- [Text Anti-Spoofing](https://huggingface.co/LaabhGupta/Text-Anti-Spoofing)

**Run this on Google Colab with a T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

Pipeline:
1. Install deps
2. Load `prithivMLmods/AI-vs-Deepfake-vs-Real` from Hugging Face (Artificial + Deepfake merged into one FAKE class vs REAL)
3. Preprocess images (resize, normalize)
4. Train & compare: BaselineCNN, DeeperCNN, ViT (transfer learning)
5. Evaluate each on a held-out test set
6. Push all models to Hugging Face Hub


## 1. Setup

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


**Important:** as we learned on the text project, `datasets` can try to import a video component from
`torchvision` which crashes on some Colab builds with `ImportError: cannot import name 'VideoReader'`.
This project only uses images. If you hit that error, run `!pip uninstall -y torchvision -q` and restart
the session, then re-run from the top (torchvision isn't strictly needed until Section 4/5, where we
reinstall what's needed for `transforms`/`models`).


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU - go to Runtime > Change runtime type > T4 GPU")


CUDA available: True
Device: Tesla T4


## 2. Load dataset

**`prithivMLmods/AI-vs-Deepfake-vs-Real`** from Hugging Face - no Kaggle API key needed.
Labels: `{0: 'Artificial', 1: 'Deepfake', 2: 'Real'}`.

We merge `Artificial` and `Deepfake` into a single `FAKE` class (label 1), keep `Real` as label 0 -
giving us the general "is this image real or AI-manipulated in any way" binary classifier you asked for.


In [4]:
from huggingface_hub import login
login()

In [6]:
from datasets import load_dataset

raw_dataset = load_dataset("prithivMLmods/AI-vs-Deepfake-vs-Real")
print(raw_dataset)


0000.parquet: reconstructing file:   0%|          |  0.00B /  508MB            

0000.parquet: downloading bytes:           |  0.00B            

0001.parquet: reconstructing file:   0%|          |  0.00B /  506MB            

0001.parquet: downloading bytes:           |  0.00B            

0002.parquet: reconstructing file:   0%|          |  0.00B /  521MB            

0002.parquet: downloading bytes:           |  0.00B            

0003.parquet: reconstructing file:   0%|          |  0.00B /  422MB            

0003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9999 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 9999
    })
})


In [7]:
# Inspect original label mapping - confirm this matches what's assumed in the next cell
print(raw_dataset["train"].features)


{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Artificial', 'Deepfake', 'Real'])}


In [11]:
from datasets import ClassLabel

LABEL_NAMES = ["REAL", "FAKE"]  # our new binary mapping

def remap_labels(example):
    original_label = example["label"]
    example["binary_label"] = 0 if original_label == 2 else 1
    return example

dataset = raw_dataset.map(remap_labels)
dataset = dataset.cast_column("binary_label", ClassLabel(names=LABEL_NAMES))  # <- new line, fixes the stratify error

print(dataset)

Map:   0%|          | 0/9999 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9999 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'binary_label'],
        num_rows: 9999
    })
})


### Optional: subsample for faster iteration

This dataset can be large. Set `SAMPLE_SIZE` below to cap how many images per split are used -
increase later for a final full-scale training run once the pipeline is confirmed working.


In [12]:
SAMPLE_SIZE = None  # e.g. set to 20000 to subsample for a quick test run, or None to use all data

if SAMPLE_SIZE:
    for split in dataset.keys():
        n = min(SAMPLE_SIZE, len(dataset[split]))
        dataset[split] = dataset[split].shuffle(seed=42).select(range(n))

print({split: len(dataset[split]) for split in dataset.keys()})


{'train': 9999}


## 3. Train / validation / test split

In [13]:
# If the dataset only ships a single 'train' split, create our own splits.
if len(dataset.keys()) == 1:
    only_split = list(dataset.keys())[0]
    split_1 = dataset[only_split].train_test_split(test_size=0.2, seed=42, stratify_by_column="binary_label")
    split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="binary_label")
    train_ds = split_1["train"]
    val_ds = split_2["train"]
    test_ds = split_2["test"]
else:
    train_ds = dataset["train"]
    val_ds = dataset["validation"] if "validation" in dataset else dataset["test"]
    test_ds = dataset["test"]

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))


Train: 7999 Val: 1000 Test: 1000


## 4. Preprocessing

In [14]:
import torchvision.transforms as T
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

IMAGE_SIZE = 224  # matches ViT-B/16 expected input size; CNNs will also use this for a fair comparison

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

class ImageAntiSpoofDataset(Dataset):
    def __init__(self, hf_dataset, image_col="image", label_col="binary_label"):
        self.dataset = hf_dataset
        self.image_col = image_col
        self.label_col = label_col

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item[self.image_col]
        if image.mode != "RGB":
            image = image.convert("RGB")
        label = item[self.label_col]
        return transform(image), torch.tensor(label, dtype=torch.long)


train_dataset = ImageAntiSpoofDataset(train_ds)
val_dataset = ImageAntiSpoofDataset(val_ds)
test_dataset = ImageAntiSpoofDataset(test_ds)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

print("Batches - train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


Batches - train: 250 val: 16 test: 16


## 5. Model architectures

Same comparative pattern as the Voice project: `BaselineCNN`, `DeeperCNN`, `ViTModel`.
Unlike voice (1-channel spectrograms), images are natively 3-channel RGB, so the ViT here
doesn't need the channel-averaging trick the voice ViT needed - it's a more standard transfer-learning setup.


In [15]:
import torch.nn as nn
import torchvision.models as models


class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        # 224 -> /2/2/2 = 28, so 64 * 28 * 28
        self.linear_stack = nn.Sequential(
            nn.Linear(64 * 28 * 28, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.linear_stack(self.flatten(self.conv_stack(x)))


class DeeperCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(), nn.BatchNorm2d(16), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.BatchNorm2d(128), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        # 224 -> /2/2/2/2 = 14, so 128 * 14 * 14
        self.linear_stack = nn.Sequential(
            nn.Linear(128 * 14 * 14, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.linear_stack(self.flatten(self.conv_stack(x)))


class ViTModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        self.vit.heads.head = nn.Linear(self.vit.heads.head.in_features, num_classes)

    def forward(self, x):
        return self.vit(x)


## 6. Training loop (shared across all three models)

In [16]:
import torch.optim as optim
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-4, model_name="model"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        start = time.time()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        elapsed = time.time() - start
        print(f"[{model_name}] Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f} - Val Acc: {val_acc:.4f} - {elapsed:.1f}s")

    return model


def evaluate_model(model, test_loader, model_name="model"):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"--- Final Test Accuracy for {model_name}: {acc:.4f} ---")
    return acc


## 7. Train Baseline CNN

In [17]:
baseline_model = BaselineCNN()
baseline_model = train_model(baseline_model, train_loader, val_loader, epochs=5, model_name="BaselineCNN")
baseline_acc = evaluate_model(baseline_model, test_loader, model_name="BaselineCNN")
torch.save(baseline_model.state_dict(), "baseline_cnn_model.pth")


[BaselineCNN] Epoch 1/5 - Loss: 0.1962 - Val Acc: 0.9590 - 104.5s
[BaselineCNN] Epoch 2/5 - Loss: 0.0753 - Val Acc: 0.9710 - 104.2s
[BaselineCNN] Epoch 3/5 - Loss: 0.0519 - Val Acc: 0.9830 - 103.4s
[BaselineCNN] Epoch 4/5 - Loss: 0.0420 - Val Acc: 0.9860 - 103.6s
[BaselineCNN] Epoch 5/5 - Loss: 0.0340 - Val Acc: 0.9900 - 103.5s
--- Final Test Accuracy for BaselineCNN: 0.9880 ---


## 8. Train Deeper CNN

In [18]:
deeper_model = DeeperCNN()
deeper_model = train_model(deeper_model, train_loader, val_loader, epochs=5, model_name="DeeperCNN")
deeper_acc = evaluate_model(deeper_model, test_loader, model_name="DeeperCNN")
torch.save(deeper_model.state_dict(), "deeper_cnn_model.pth")


[DeeperCNN] Epoch 1/5 - Loss: 0.0848 - Val Acc: 0.9790 - 102.7s
[DeeperCNN] Epoch 2/5 - Loss: 0.0195 - Val Acc: 0.9880 - 102.2s
[DeeperCNN] Epoch 3/5 - Loss: 0.0108 - Val Acc: 0.9970 - 102.8s
[DeeperCNN] Epoch 4/5 - Loss: 0.0039 - Val Acc: 0.9950 - 103.0s
[DeeperCNN] Epoch 5/5 - Loss: 0.0020 - Val Acc: 0.9940 - 102.3s
--- Final Test Accuracy for DeeperCNN: 0.9920 ---


## 9. Train ViT

In [19]:
vit_model = ViTModel()
vit_model = train_model(vit_model, train_loader, val_loader, epochs=3, lr=3e-5, model_name="ViT")  # ViT needs fewer epochs, lower LR
vit_acc = evaluate_model(vit_model, test_loader, model_name="ViT")
torch.save(vit_model.state_dict(), "vit_model.pth")


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:05<00:00, 58.2MB/s]


[ViT] Epoch 1/3 - Loss: 0.0183 - Val Acc: 1.0000 - 312.7s
[ViT] Epoch 2/3 - Loss: 0.0049 - Val Acc: 1.0000 - 311.6s
[ViT] Epoch 3/3 - Loss: 0.0001 - Val Acc: 1.0000 - 311.5s
--- Final Test Accuracy for ViT: 1.0000 ---


## 10. Comparison

In [20]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Baseline CNN", "Deeper CNN", "ViT"],
    "Test Accuracy": [baseline_acc, deeper_acc, vit_acc],
})
print(results.sort_values("Test Accuracy", ascending=False))


          Model  Test Accuracy
2           ViT          1.000
1    Deeper CNN          0.992
0  Baseline CNN          0.988


## 11. Push all models to Hugging Face Hub

Upload all three regardless of which wins - keeps the comparison transparent, same as the voice repo.


In [22]:
from huggingface_hub import  upload_file

HF_REPO_ID = "LaabhGupta/image-antispoofing"  # create this repo on huggingface.co/new first

upload_file(path_or_fileobj="baseline_cnn_model.pth", path_in_repo="baseline_cnn_model.pth", repo_id=HF_REPO_ID)
upload_file(path_or_fileobj="deeper_cnn_model.pth", path_in_repo="deeper_cnn_model.pth", repo_id=HF_REPO_ID)
upload_file(path_or_fileobj="vit_model.pth", path_in_repo="vit_model.pth", repo_id=HF_REPO_ID)

print(f"Uploaded all three models to https://huggingface.co/{HF_REPO_ID}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/baseline_cnn_model.pth:   2%|2         |  563kB / 25.8MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tent/deeper_cnn_model.pth:   2%|2         |  566kB / 26.1MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/vit_model.pth      :   0%|          |  547kB /  343MB            

Uploaded all three models to https://huggingface.co/LaabhGupta/image-antispoofing


---
### Next steps
1. Confirm the comparison table above - note which model actually wins (don't assume ViT automatically wins, per the voice project's lesson)
2. Write `model.py` (architecture classes + `preprocess_image()`) and a proper README/model card - same pattern as voice
3. Build a FastAPI backend with an image-upload `/predict/` endpoint, mirroring the text/voice backend pattern
4. Deploy backend to Render, frontend to Netlify - same playbook as before
